# vehicle2: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('32_vehicle2_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,96.034,95.075,97.131,93.118,99.046,96.082,98.821,99.536,95.555
smote,96.774,95.283,95.445,95.185,98.408,96.797,98.882,99.572,96.028
bsmote,97.340,95.065,93.010,97.241,97.451,97.346,98.466,99.446,96.203
ros,96.852,95.505,95.890,95.180,98.568,96.874,98.902,99.598,96.178
adasyn,97.383,95.565,94.421,96.781,98.009,97.395,98.921,99.604,96.474
casb,96.929,95.312,95.052,95.645,98.250,96.947,98.825,99.536,96.120
casb_relaxed,97.359,96.094,96.139,96.099,98.649,97.374,98.746,99.524,96.726
casb_edited,96.725,94.385,92.791,96.094,97.373,96.733,98.630,99.486,95.555
casb_ros,96.893,95.203,94.821,95.645,98.171,96.908,98.751,99.514,96.048


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.560,92.572,90.758,94.493,96.655,95.574,98.572,99.436,94.066
smote,95.372,90.914,86.322,96.089,94.670,95.379,98.123,99.411,93.143
bsmote,95.371,89.611,82.575,98.177,92.679,95.428,96.953,99.256,92.491
ros,95.509,90.843,85.766,96.797,94.269,95.533,97.498,99.353,93.176
adasyn,95.700,90.591,84.491,97.933,93.556,95.744,97.498,99.318,93.145
casb,95.597,90.623,84.862,97.468,93.790,95.629,98.260,99.485,93.110
casb_relaxed,95.556,90.937,85.892,96.786,94.353,95.570,98.067,99.440,93.246
casb_edited,95.412,90.307,84.441,97.241,93.634,95.438,95.816,99.051,92.859
casb_ros,95.619,90.909,85.616,97.003,94.268,95.636,98.340,99.434,93.264


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,97.905,97.209,97.944,96.575,99.286,97.930,99.511,99.802,97.557
smote,98.141,97.241,97.277,97.262,99.046,98.154,99.536,99.818,97.691
bsmote,98.411,97.377,96.856,97.955,98.886,98.420,99.608,99.844,97.894
ros,98.219,97.249,97.043,97.495,98.966,98.230,99.576,99.836,97.734
adasyn,98.021,96.909,96.621,97.267,98.807,98.037,99.586,99.831,97.465
casb,97.670,96.543,96.567,96.570,98.807,97.688,99.637,99.853,97.107
casb_relaxed,97.904,96.781,96.587,97.035,98.807,97.921,99.552,99.822,97.343
casb_edited,97.826,96.579,96.204,97.030,98.648,97.839,99.565,99.824,97.203
casb_ros,98.179,97.137,96.822,97.495,98.886,98.191,99.555,99.827,97.658


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,93.745,91.170,92.089,90.359,97.295,93.827,94.752,98.433,92.457
smote,95.322,91.956,89.674,94.514,96.180,95.347,94.924,98.478,93.639
bsmote,95.197,91.475,88.575,94.730,95.703,95.217,95.645,98.492,93.336
ros,95.086,91.711,89.624,94.043,96.180,95.112,95.014,98.524,93.398
adasyn,95.977,92.568,89.511,95.862,96.100,95.981,95.536,98.508,94.272
casb,95.274,91.854,89.504,94.498,96.100,95.299,95.279,98.539,93.564
casb_relaxed,95.438,92.077,89.682,94.736,96.180,95.458,95.214,98.543,93.757
casb_edited,95.005,90.976,87.583,94.725,95.305,95.015,93.578,98.206,92.991
casb_ros,95.189,91.640,89.122,94.498,95.942,95.220,95.269,98.548,93.414


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.811,94.006,94.480,93.636,98.071,95.853,97.914,99.302,94.909
smote,96.402,93.848,92.180,95.762,97.076,96.419,97.866,99.320,95.125
bsmote,96.580,93.382,90.254,97.026,96.180,96.603,97.668,99.259,94.981
ros,96.416,93.827,92.081,95.879,96.996,96.437,97.748,99.328,95.122
adasyn,96.770,93.908,91.261,96.961,96.618,96.789,97.885,99.315,95.339
casb,96.367,93.583,91.496,96.045,96.737,96.391,98.000,99.353,94.975
casb_relaxed,96.564,93.972,92.075,96.164,96.997,96.581,97.895,99.332,95.268
casb_edited,96.242,93.062,90.255,96.272,96.240,96.256,96.897,99.142,94.652
casb_ros,96.470,93.722,91.595,96.160,96.817,96.488,97.978,99.331,95.096


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,97.359,adasyn,97.383,-0.024,LOSS
1,f1,casb_relaxed,96.094,adasyn,95.565,0.529,WIN
2,precision,casb_relaxed,96.139,ros,95.890,0.249,WIN
3,recall,casb_relaxed,96.099,bsmote,97.241,-1.142,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,95.619,adasyn,95.700,-0.080,LOSS
1,f1,casb_relaxed,90.937,smote,90.914,0.024,WIN
2,precision,casb_relaxed,85.892,smote,86.322,-0.430,LOSS
3,recall,casb,97.468,bsmote,98.177,-0.708,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,98.179,bsmote,98.411,-0.231,LOSS
1,f1,casb_ros,97.137,bsmote,97.377,-0.240,LOSS
2,precision,casb_ros,96.822,smote,97.277,-0.455,LOSS
3,recall,casb_ros,97.495,bsmote,97.955,-0.460,LOSS


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,96.564,adasyn,96.770,-0.206,LOSS
1,f1,casb_relaxed,93.972,adasyn,93.908,0.064,WIN
2,precision,casb_relaxed,92.075,smote,92.180,-0.104,LOSS
3,recall,casb_edited,96.272,bsmote,97.026,-0.753,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.949671,0.941176,0.975610,0.909091,0.992063,0.950577,0.980108,0.992785,0.945424,None
1,42,1,svm,smote,0.75,0.957615,0.942529,0.953488,0.931818,0.984127,0.957973,0.979392,0.993146,0.950072,None
2,42,1,svm,bsmote,0.50,0.969223,0.954545,0.954545,0.954545,0.984127,0.969336,0.984940,0.994769,0.961884,None
3,42,1,svm,ros,0.50,0.949671,0.941176,0.975610,0.909091,0.992063,0.950577,0.982466,0.993867,0.945424,None
4,42,1,svm,adasyn,0.50,0.957615,0.942529,0.953488,0.931818,0.984127,0.957973,0.979807,0.993687,0.950072,None
5,42,1,svm,casb,1.00,0.961375,0.933333,0.913043,0.954545,0.968254,0.961400,0.983179,0.994769,0.947354,None
6,42,1,svm,casb_relaxed,0.75,0.969223,0.954545,0.954545,0.954545,0.984127,0.969336,0.977067,0.992605,0.961884,None
7,42,1,svm,casb_edited,1.00,0.964747,0.924731,0.877551,0.977273,0.952381,0.964827,0.978279,0.993326,0.944739,None
8,42,1,svm,casb_ros,0.75,0.957615,0.942529,0.953488,0.931818,0.984127,0.957973,0.977123,0.992965,0.950072,None
9,42,1,knn,original,0.50,0.953746,0.931818,0.931818,0.931818,0.976190,0.954004,0.985055,0.993506,0.942782,None
